In [ ]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    r"G:\My Drive\Spacesmith and Wordsmith's Tower\Spacesmith's HQ"
    r"\Nuclear Energy and Propulsion Engineering\Accenture AI-ML Computational Scientist"
    r"\Credentials\gen-lang-client-0137385761-b0d89e37e8e5.json"
)

In [ ]:
from google.cloud import bigquery
from google.oauth2 import service_account

# Google Cloud credentials
credentials = service_account.Credentials.from_service_account_file(

    r"G:\My Drive\Spacesmith and Wordsmith's Tower\Spacesmith's HQ\Nuclear Energy and Propulsion Engineering\Accenture AI-ML Computational Scientist\Credentials\gen-lang-client-0137385761-b0d89e37e8e5.json"

)

In [ ]:
# Create a "Client" object
client = bigquery.Client()

In [ ]:
# Strategy 4: Filter on the small table FIRST, then join the large table
#
# The goal: find the most recent GPS location for each costume owned by MitzieOwnerID.
#
# CostumeLocations has GPS pings every second for every costume — billions of rows.
# CostumeOwners is tiny: just costume-to-owner mappings.
#
# The slow way joins ALL owners with ALL location rows first, then filters at the end.
# The fast way filters down to just Mitzie's costumes on the small table first,
# so the expensive CostumeLocations scan only touches the few rows that matter.

slow_query = """
             WITH LocationsAndOwners AS (
                 -- Joins EVERY owner with EVERY location row before any filtering.
                 -- This creates a massive intermediate table — billions of rows.
                 SELECT *
                 FROM CostumeOwners co
                 INNER JOIN CostumeLocations cl
                     ON co.CostumeID = cl.CostumeID
             ),
             LastSeen AS (
                 -- Finds the most recent timestamp per costume from the giant joined table.
                 SELECT CostumeID, MAX(Timestamp) AS MaxTimestamp
                 FROM LocationsAndOwners
                 GROUP BY CostumeID
             )
             -- Finally filters by MitzieOwnerID — but the expensive work is already done.
             SELECT lo.CostumeID, Location
             FROM LocationsAndOwners lo
             INNER JOIN LastSeen ls
                 ON lo.Timestamp = ls.MaxTimestamp
                 AND lo.CostumeID = ls.CostumeID
             WHERE OwnerID = MitzieOwnerID
             """
show_amount_of_data_scanned(slow_query)

fast_query = """
             WITH MitziesCostumes AS (
                 -- Step 1: Filter the small CostumeOwners table FIRST.
                 -- This gives us only the CostumeIDs that belong to MitzieOwnerID.
                 -- CostumeOwners is tiny, so this is nearly free.
                 SELECT CostumeID
                 FROM CostumeOwners
                 WHERE OwnerID = MitzieOwnerID
             ),
             LastSeen AS (
                 -- Step 2: Join CostumeLocations against ONLY Mitzie's costume IDs.
                 -- BigQuery now scans a tiny fraction of the billions of location rows.
                 -- MAX(Timestamp) finds the most recent ping per costume.
                 SELECT cl.CostumeID, MAX(cl.Timestamp) AS MaxTimestamp
                 FROM CostumeLocations cl
                 INNER JOIN MitziesCostumes mc
                     ON cl.CostumeID = mc.CostumeID
                 GROUP BY cl.CostumeID
             )
             -- Step 3: Join back to CostumeLocations to get the actual GPS coordinates
             -- at the most recent timestamp for each of Mitzie's costumes.
             SELECT ls.CostumeID, cl.Location
             FROM CostumeLocations cl
             INNER JOIN LastSeen ls
                 ON cl.CostumeID = ls.CostumeID
                 AND cl.Timestamp = ls.MaxTimestamp
             """
show_amount_of_data_scanned(fast_query)
